## Imports

In [159]:
import json
import re
import os
import time
import warnings
from pathlib import Path
from tqdm.auto import tqdm

import torch
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from senticnet.senticnet import SenticNet
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings('ignore')
print(f"PyTorch version : {torch.__version__}")
print(f"GPU available : {torch.cuda.is_available()}")
DEVICE = 0 if torch.cuda.is_available() else -1

PyTorch version : 2.10.0+cu126
GPU available : True


## Config

In [160]:
INPUT_FILES = [
    "../final_dataset/raw_data.json"
    # "../dataset/reddit.json"
]
OUTPUT_FILE = "../final_dataset/classified_output.json"
# OUTPUT_FILE = "../final_dataset/reddit.json"
# OUTPUT_FILE = "../dataset/combined_output.json"


# Routing thresholds 
SHORT_WORD_LIMIT = 60    # <= SHORT_WORD_LIMIT  → VADER-primary path
LONG_WORD_LIMIT = 400   # >= LONG_WORD_LIMIT   → chunked transformer path
                          # between the two      → transformer direct path

# Subjectivity 
TEXTBLOB_SUBJ_THRESHOLD = 0.20   # below this → objective
VADER_NEUTRAL_BAND = 0.05   # abs(compound) < this → likely objective

# Ensemble weights (must sum to 1.0) for [VADER, SenticNet, Transformer]
SHORT_WEIGHTS= [0.35, 0.20, 0.45]
MEDIUM_WEIGHTS = [0.20, 0.30, 0.50]
LONG_WEIGHTS = [0.10, 0.35, 0.55]

# Comment context 
# Max words taken from parent text and prepended to the comment before classify
PARENT_CONTEXT_WORDS = 60

# Transformer model 
TRANSFORMER_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
MAX_TOKENS = 512
TRANSFORMER_BATCH = 32

print("Configuration loaded.")

Configuration loaded.


## Load Models

In [161]:
# VADER
vader = SentimentIntensityAnalyzer()

# SenticNet
sn = SenticNet()

# load transformer once, reused for all records
print(f"Loading transformer: {TRANSFORMER_MODEL} ...")
tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL)
transformer = pipeline(
    "sentiment-analysis",
    model=TRANSFORMER_MODEL,
    tokenizer=tokenizer,
    device=DEVICE,
    truncation=True,
    max_length=MAX_TOKENS,
    top_k=None,           # return all label scores
)
print("All models loaded.")

Loading transformer: cardiffnlp/twitter-roberta-base-sentiment-latest ...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1350.49it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All models loaded.


## Domain-Specific (Specific to vibe coding) Lexicon Corrections

VADER and SenticNet both lack awareness of emerging "vibe coding" vocabulary.
These corrections apply **after** the base models score the text.

In [162]:
# Each entry: (regex_pattern, polarity_nudge)
# nudge is in [-1, 1]; added to the normalised score before clamping
DOMAIN_CORRECTIONS = [
    # Negative concepts
    (r"\bspaghetti code\b", -0.30),
    (r"\btechnical debt\b", -0.25),
    (r"\bskill rot\b", -0.30),
    (r"\bhallucina(te|ting|tion)\b", -0.25),
    (r"\bsecurity (risk|vulnerabilit)", -0.25),
    (r"\bimposter syndrome\b", -0.20),
    (r"\bunmaintainable\b", -0.25),
    (r"\bcode (rot|smell)\b", -0.20),
    (r"\bdeath of (junior|dev)\b", -0.20),
    # Positive concepts
    (r"\b10x (productiv|faster)\b", +0.25),
    (r"\brapid prototyp\b", +0.15),
    (r"\brevolution(ary)?\b", +0.15),
    (r"\bgame.?changer\b", +0.20),
    (r"\blove (vibe|cursor|windsurf)\b",+0.25),
]

def apply_domain_corrections(text: str, base_score: float) -> float:
    """Nudge a [-1,1] polarity score using domain-specific patterns."""
    text_lower = text.lower()
    nudge = 0.0
    for pattern, delta in DOMAIN_CORRECTIONS:
        if re.search(pattern, text_lower):
            nudge += delta
    return max(-1.0, min(1.0, base_score + nudge))

print(f"Domain corrections: {len(DOMAIN_CORRECTIONS)} rules loaded.")

Domain corrections: 14 rules loaded.


## Layer 1: Rule-Based Subjectivity Detection

In [163]:
FIRST_PERSON = re.compile(r"\b(i|me|my|mine|myself|we|our|ours|ourselves)\b", re.I)
HEDGING = re.compile(r"\b(think|believe|feel|seems?|probably|maybe|perhaps|in my opinion|imo|imho|personally)\b", re.I)
OPINION_MARKERS = re.compile(r"\b(love|hate|great|terrible|awful|amazing|horrible|best|worst|favourite|prefer)\b", re.I)

# Sources that are almost always opinionated
SUBJECTIVE_SOURCES = {"reddit", "twitter", "x"}

def detect_subjectivity(text: str, source: str, word_count: int) -> dict:
    """
    Returns:
        label  : 'subjective' | 'objective'
        score  : float in [0, 1] — higher = more subjective
        method : which signals fired
    """
    signals = []
    score = 0.0

    # Signal 1 — TextBlob subjectivity (0 = objective, 1 = subjective)
    tb_score = TextBlob(text).sentiment.subjectivity
    score += tb_score * 0.40          # 40% weight
    if tb_score > TEXTBLOB_SUBJ_THRESHOLD:
        signals.append("textblob")

    # Signal 2 — First-person pronoun density
    word_count_safe = max(word_count, 1)
    fp_density = len(FIRST_PERSON.findall(text)) / word_count_safe
    score += min(fp_density * 5, 1.0) * 0.20   # cap density contribution
    if fp_density > 0.02:
        signals.append("first_person")

    # Signal 3 — Hedging / opinion markers
    has_hedging = bool(HEDGING.search(text))
    has_opinion = bool(OPINION_MARKERS.search(text))
    if has_hedging:
        score += 0.15
        signals.append("hedging")
    if has_opinion:
        score += 0.15
        signals.append("opinion_markers")

    # Signal 4 — VADER neutral band check (very neutral = likely objective)
    vader_compound = vader.polarity_scores(text)["compound"]
    if abs(vader_compound) < VADER_NEUTRAL_BAND:
        score -= 0.10   # slight pull toward objective
    
    # Signal 5 — Source prior
    if source.lower() in SUBJECTIVE_SOURCES:
        score += 0.10
        signals.append("source_prior")

    score = max(0.0, min(1.0, score))
    label = "subjective" if score >= 0.30 else "objective"

    return {
        "Subjectivity"        : label,
        "Subjectivity_Score"  : round(score, 4),
        "Subjectivity_Signals": signals,
    }

# Quick sanity check
_test = detect_subjectivity("I think vibe coding is absolutely terrible for code quality.", "reddit", 12)
print("Subjectivity test:", _test)

Subjectivity test: {'Subjectivity': 'subjective', 'Subjectivity_Score': 0.8833, 'Subjectivity_Signals': ['textblob', 'first_person', 'hedging', 'opinion_markers', 'source_prior']}


## Layer 2a: VADER (Knowledge-Based)

In [164]:
def score_vader(text: str) -> dict:
    """
    Returns compound score in [-1, 1] and a label.
    Domain corrections applied on top.
    """
    scores  = vader.polarity_scores(text)
    compound = scores["compound"]
    compound = apply_domain_corrections(text, compound)

    if compound >= 0.05:
        label = "positive"
    elif compound <= -0.05:
        label = "negative"
    else:
        label = "neutral"

    return {
        "VADER_Compound": round(compound, 4),
        "VADER_Label"   : label,
    }

print("VADER test:", score_vader("Vibe coding is amazing, I love it!"))

VADER test: {'VADER_Compound': 0.8516, 'VADER_Label': 'positive'}


## Layer 2b: SenticNet (Knowledge-Based)

In [165]:
def _get_sn_polarity(concept: str) -> float | None:
    """Lookup a single concept in SenticNet. Returns None if not found."""
    try:
        data = sn.concept(concept.lower().replace(" ", "_"))
        return float(data["polarity_value"])
    except Exception:
        return None

def score_senticnet(text: str) -> dict:
    """
    Extracts unigrams and bigrams, looks each up in SenticNet,
    and returns the mean polarity of recognised concepts.
    Falls back to 0.0 (neutral) if nothing is recognised.
    """
    words   = re.findall(r"[a-zA-Z]+", text.lower())
    # Unigrams + bigrams as candidate concepts
    concepts = words + [f"{words[i]}_{words[i+1]}" for i in range(len(words) - 1)]

    polarities = []
    hit_concepts = []
    for c in concepts:
        p = _get_sn_polarity(c)
        if p is not None:
            polarities.append(p)
            hit_concepts.append(c)

    if polarities:
        mean_polarity = sum(polarities) / len(polarities)
    else:
        mean_polarity = 0.0  # no coverage → neutral

    mean_polarity = apply_domain_corrections(text, mean_polarity)

    if mean_polarity >= 0.05:
        label = "positive"
    elif mean_polarity <= -0.05:
        label = "negative"
    else:
        label = "neutral"

    return {
        "SenticNet_Score"    : round(mean_polarity, 4),
        "SenticNet_Label"    : label,
        "SenticNet_Coverage" : len(polarities),   # how many concepts were found
    }

print("SenticNet test:", score_senticnet("This produces terrible technical debt and spaghetti code."))

SenticNet test: {'SenticNet_Score': -0.7743, 'SenticNet_Label': 'negative', 'SenticNet_Coverage': 4}


## Layer 3: Transformer (ML-Based)
- Short/medium texts: passed directly (truncated at 512 tokens).
- Long texts: split into paragraph chunks, each classified, then aggregated.

In [166]:
# Map whatever labels the model returns → our standard three labels
LABEL_MAP = {
    "positive" : "positive",
    "negative" : "negative",
    "neutral"  : "neutral",
    "label_0"  : "negative",   # some models use numeric labels
    "label_1"  : "neutral",
    "label_2"  : "positive",
}

def _parse_transformer_output(raw: list[dict]) -> dict:
    """
    raw is the top_k=None output: [{label, score}, ...]
    Returns {label, confidence, pos_score, neg_score, neu_score}
    """
    scores = {LABEL_MAP.get(r["label"].lower(), r["label"].lower()): r["score"] for r in raw}
    pos = scores.get("positive", 0.0)
    neg = scores.get("negative", 0.0)
    neu = scores.get("neutral",  0.0)
    label = max(scores, key=scores.get)
    return {
        "Transformer_Label": label,
        "Transformer_Confidence": round(scores[label], 4),
        "Transformer_Pos": round(pos, 4),
        "Transformer_Neg": round(neg, 4),
        "Transformer_Neu": round(neu, 4),
    }

def _chunk_text(text: str, max_words: int = 300) -> list[str]:
    """Split long text into paragraph-aware chunks of ≤max_words words."""
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    if not paragraphs:
        paragraphs = [text]

    chunks, current, current_wc = [], [], 0
    for para in paragraphs:
        wc = len(para.split())
        if current_wc + wc > max_words and current:
            chunks.append(" ".join(current))
            current, current_wc = [para], wc
        else:
            current.append(para)
            current_wc += wc
    if current:
        chunks.append(" ".join(current))
    return chunks

def score_transformer(text: str, word_count: int) -> dict:
    """
    Route text through the transformer.
    Long texts are chunked-> scores are aggregated by confidence-weighted average.
    """
    if word_count < LONG_WORD_LIMIT:
        # Direct pass
        raw = transformer(text)[0]   # top_k=None -> list of dicts
        result = _parse_transformer_output(raw)
        result["Transformer_Chunks"] = 1
        return result

    # Chunked pass for long documents
    chunks = _chunk_text(text)
    chunk_results = []
    for chunk in chunks:
        raw = transformer(chunk)[0]
        chunk_results.append(_parse_transformer_output(raw))

    # Aggregate: confidence-weighted average of pos/neg/neu scores
    total_conf = sum(r["Transformer_Confidence"] for r in chunk_results)
    agg_pos = sum(r["Transformer_Pos"] * r["Transformer_Confidence"] for r in chunk_results) / total_conf
    agg_neg = sum(r["Transformer_Neg"] * r["Transformer_Confidence"] for r in chunk_results) / total_conf
    agg_neu = sum(r["Transformer_Neu"] * r["Transformer_Confidence"] for r in chunk_results) / total_conf

    agg_scores = {"positive": agg_pos, "negative": agg_neg, "neutral": agg_neu}
    agg_label  = max(agg_scores, key=agg_scores.get)

    return {
        "Transformer_Label": agg_label,
        "Transformer_Confidence": round(agg_scores[agg_label], 4),
        "Transformer_Pos": round(agg_pos, 4),
        "Transformer_Neg": round(agg_neg, 4),
        "Transformer_Neu": round(agg_neu, 4),
        "Transformer_Chunks": len(chunks),
    }

print("Transformer ready.")

Transformer ready.


## Ensemble & Routing Logic

In [167]:
LABEL_TO_SCORE = {"positive": 1.0, "neutral": 0.0, "negative": -1.0}
SCORE_THRESHOLDS = {"positive": 0.15, "negative": -0.15}  # inside → neutral

def route_and_classify(text: str, word_count: int) -> str:
    """Return 'short', 'medium', or 'long' based on word count."""
    if word_count <= SHORT_WORD_LIMIT:
        return "short"
    elif word_count >= LONG_WORD_LIMIT:
        return "long"
    return "medium"

def ensemble_polarity(
    vader_result      : dict,
    senticnet_result  : dict,
    transformer_result: dict,
    route             : str,
) -> dict:
    """
    Weighted average of the three model scores → final label.
    Weights are route-dependent.
    """
    weights = {
        "short" : SHORT_WEIGHTS,
        "medium": MEDIUM_WEIGHTS,
        "long"  : LONG_WEIGHTS,
    }[route]

    v_score  = LABEL_TO_SCORE.get(vader_result["VADER_Label"],        0.0)
    sn_score = LABEL_TO_SCORE.get(senticnet_result["SenticNet_Label"],0.0)

    # For the transformer we use its raw pos/neg/neu probabilities
    t_score = (
        transformer_result["Transformer_Pos"] -
        transformer_result["Transformer_Neg"]
    )  # range approx [-1, 1]

    # SenticNet: if coverage is very low, down-weight it and reallocate to VADER
    sn_coverage = senticnet_result.get("SenticNet_Coverage", 0)
    if sn_coverage < 2:
        # redistribute SenticNet weight to VADER
        w_v  = weights[0] + weights[1] * 0.7
        w_sn = weights[1] * 0.3
        w_t  = weights[2]
    else:
        w_v, w_sn, w_t = weights

    final_score = w_v * v_score + w_sn * sn_score + w_t * t_score

    if final_score >= SCORE_THRESHOLDS["positive"]:
        final_label = "positive"
    elif final_score <= SCORE_THRESHOLDS["negative"]:
        final_label = "negative"
    else:
        final_label = "neutral"

    return {
        "Polarity"         : final_label,
        "Polarity_Score"   : round(final_score, 4),
        "Ensemble_Route"   : route,
        "Ensemble_Weights" : {"vader": round(w_v,3), "senticnet": round(w_sn,3), "transformer": round(w_t,3)},
    }

print("Ensemble logic ready.")

Ensemble logic ready.


##  Master Classify Function
Accepts the text to classify and an optional `context_text` (used for comments).

In [168]:
def classify(text: str, source: str, word_count: int, context_text: str | None = None) -> dict:
    """
    Parameters
    text         : The raw text to classify.
    source       : Platform source string (e.g. 'Reddit').
    word_count   : Pre-computed word count of `text` alone.
    context_text : Parent post/comment text (for comments). When provided,
                   a truncated version is prepended to `text` so the model
                   understands the conversational context.

    Returns
    dict of all classification fields to be merged into the record.
    """
    # Build classification text 
    if context_text:
        context_words = " ".join(str(context_text).split()[:PARENT_CONTEXT_WORDS])
        # Delimiter makes it clear to the model where context ends / reply begins
        classify_text = f"[CONTEXT]: {context_words} [REPLY]: {text}"
    else:
        classify_text = text

    classify_wc = len(classify_text.split())

    # Layer 1: Subjectivity 
    subjectivity = detect_subjectivity(classify_text, source, classify_wc)

    # If objective, skip polarity — return early with neutral defaults
    if subjectivity["Subjectivity"] == "objective":
        return {
            **subjectivity,
            "Polarity": "neutral",
            "Polarity_Score": 0.0,
            "Ensemble_Route": "skipped_objective",
            "Ensemble_Weights": {},
            "VADER_Compound": 0.0,
            "VADER_Label": "neutral",
            "SenticNet_Score": 0.0,
            "SenticNet_Label": "neutral",
            "SenticNet_Coverage": 0,
            "Transformer_Label": "neutral",
            "Transformer_Confidence": 0.0,
            "Transformer_Pos": 0.0,
            "Transformer_Neg": 0.0,
            "Transformer_Neu": 1.0,
            "Transformer_Chunks": 0,
        }

    # Route based on length 
    route = route_and_classify(classify_text, classify_wc)

    # Layer 2: Knowledge-Based 
    vader_result = score_vader(classify_text)
    senticnet_result = score_senticnet(classify_text)

    # Layer 3: Transformer 
    transformer_result = score_transformer(classify_text, classify_wc)

    # Ensemble
    ensemble = ensemble_polarity(vader_result, senticnet_result, transformer_result, route)

    return {
        **subjectivity,
        **vader_result,
        **senticnet_result,
        **transformer_result,
        **ensemble,
    }

# sanity check
_post_text = "I've been vibe coding for 3 months and my productivity has gone through the roof!"
_comment_text = "Agree, but the spaghetti code is a nightmare to maintain."

print("Post result:")
print(json.dumps(classify(_post_text, "reddit", len(_post_text.split())), indent=2))

print("\nComment result (with parent context):")
print(json.dumps(classify(_comment_text, "reddit", len(_comment_text.split()), context_text=_post_text), indent=2))

Post result:
{
  "Subjectivity": "objective",
  "Subjectivity_Score": 0.1333,
  "Subjectivity_Signals": [
    "first_person",
    "source_prior"
  ],
  "Polarity": "neutral",
  "Polarity_Score": 0.0,
  "Ensemble_Route": "skipped_objective",
  "Ensemble_Weights": {},
  "VADER_Compound": 0.0,
  "VADER_Label": "neutral",
  "SenticNet_Score": 0.0,
  "SenticNet_Label": "neutral",
  "SenticNet_Coverage": 0,
  "Transformer_Label": "neutral",
  "Transformer_Confidence": 0.0,
  "Transformer_Pos": 0.0,
  "Transformer_Neg": 0.0,
  "Transformer_Neu": 1.0,
  "Transformer_Chunks": 0
}

Comment result (with parent context):
{
  "Subjectivity": "objective",
  "Subjectivity_Score": 0.1741,
  "Subjectivity_Signals": [
    "first_person",
    "source_prior"
  ],
  "Polarity": "neutral",
  "Polarity_Score": 0.0,
  "Ensemble_Route": "skipped_objective",
  "Ensemble_Weights": {},
  "VADER_Compound": 0.0,
  "VADER_Label": "neutral",
  "SenticNet_Score": 0.0,
  "SenticNet_Label": "neutral",
  "SenticNet_Cover

## Load JSON Files

In [169]:
all_posts = []

for file_path in INPUT_FILES:
    p = Path(file_path)
    if not p.exists():
        print(f"[WARN] File not found, skipping: {file_path}")
        continue
    with open(p, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"Loaded {len(data):,} posts from {p.name}")
    all_posts.extend(data)

print(f"\nTotal posts to process: {len(all_posts):,}")

total_comments = sum(len(p.get("Comments") or []) for p in all_posts)
print(f"Total comments to process: {total_comments:,}")
print(f"Grand total records: {len(all_posts) + total_comments:,}")

Loaded 47,444 posts from raw_data.json

Total posts to process: 47,444
Total comments to process: 37,672
Grand total records: 85,116


## Main Pipeline

In [170]:
output_posts = []
errors = []

start_time = time.time()

for post in tqdm(all_posts, desc="Processing posts"):
    post_out = dict(post)   # shallow copy; we'll update it
    post_text = str(post.get("Text", ""))
    post_source = str(post.get("Source", "unknown"))
    post_wc = int(post.get("Word_Count", len(post_text.split())))

    # Classify the post 
    try:
        post_sentiment = classify(post_text, post_source, post_wc, context_text=None)
        post_out.update(post_sentiment)
    except Exception as e:
        errors.append({"id": post.get("ID"), "level": "post", "error": str(e)})
        # Set safe defaults so the post still makes it into the output
        post_out.update({
            "Subjectivity": "error", "Polarity": "error",
            "Error": str(e)
        })

    # classify comment (note that comments must use parent's text as context)
    classified_comments = []
    for comment in post.get("Comments") or []:
        comment_out = dict(comment)
        comment_text = str(comment.get("Text", ""))
        comment_source = str(comment.get("Source", post_source))
        comment_wc = int(comment.get("Word_Count", len(comment_text.split())))

        # Pass parent post text as context for the comment
        try:
            comment_sentiment = classify(
                comment_text,
                comment_source,
                comment_wc,
                context_text=post_text,   # ← parent's textual context injected here
            )
            comment_out.update(comment_sentiment)
        except Exception as e:
            errors.append({
                "id"   : comment.get("comment_id"),
                "level": "comment",
                "error": str(e)
            })
            comment_out.update({
                "Subjectivity": "error", "Polarity": "error",
                "Error": str(e)
            })

        classified_comments.append(comment_out)

    post_out["Comments"] = classified_comments
    output_posts.append(post_out)

elapsed = time.time() - start_time
total_records = len(all_posts) + total_comments

print(f"\n Processing Stats:")
print(f"Total time: {elapsed:.1f}s")
print(f"Records processed: {total_records:,}")
print(f"Records/second: {total_records / elapsed:.1f}")
print(f"Errors: {len(errors)}")

Processing posts: 100%|██████████| 47444/47444 [31:29<00:00, 25.10it/s]   


 Processing Stats:
Total time: 1889.8s
Records processed: 85,116
Records/second: 45.0
Errors: 0


## Save Output

In [171]:
Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(output_posts, f, indent=2, ensure_ascii=False)

print(f"Saved {len(output_posts):,} classified posts → {OUTPUT_FILE}")

if errors:
    error_path = OUTPUT_FILE.replace(".json", "_errors.json")
    with open(error_path, "w", encoding="utf-8") as f:
        json.dump(errors, f, indent=2)
    print(f"Saved {len(errors)} errors → {error_path}")

Saved 47,444 classified posts → ../final_dataset/classified_output.json


## Stats

In [172]:
from collections import Counter

post_polarities = Counter()
post_subjectivities = Counter()
comment_polarities = Counter()
comment_subj = Counter()
route_counts = Counter()

for post in output_posts:
    post_polarities[post.get("Polarity", "unknown")] += 1
    post_subjectivities[post.get("Subjectivity", "unknown")] += 1
    route_counts[post.get("Ensemble_Route", "unknown")] += 1
    for comment in post.get("Comments", []):
        comment_polarities[comment.get("Polarity", "unknown")] += 1
        comment_subj[comment.get("Subjectivity", "unknown")] += 1
        route_counts[comment.get("Ensemble_Route", "unknown")] += 1

print("── POST SUBJECTIVITY ─────────────────────────")
for k, v in post_subjectivities.most_common():
    print(f"  {k:<15}: {v:>7,}")

print("── POST POLARITY ─────────────────────────────")
for k, v in post_polarities.most_common():
    print(f"  {k:<15}: {v:>7,}")

print("── COMMENT SUBJECTIVITY ──────────────────────")
for k, v in comment_subj.most_common():
    print(f"  {k:<15}: {v:>7,}")

print("── COMMENT POLARITY ──────────────────────────")
for k, v in comment_polarities.most_common():
    print(f"  {k:<15}: {v:>7,}")

print("── ROUTING DISTRIBUTION ──────────────────────")
for k, v in route_counts.most_common():
    print(f"  {k:<25}: {v:>7,}")

── POST SUBJECTIVITY ─────────────────────────
  subjective     :  24,438
  objective      :  23,006
── POST POLARITY ─────────────────────────────
  neutral        :  25,079
  positive       :  17,862
  negative       :   4,503
── COMMENT SUBJECTIVITY ──────────────────────
  subjective     :  27,607
  objective      :  10,065
── COMMENT POLARITY ──────────────────────────
  positive       :  20,253
  neutral        :  13,425
  negative       :   3,994
── ROUTING DISTRIBUTION ──────────────────────
  skipped_objective        :  33,071
  medium                   :  31,113
  short                    :  15,288
  long                     :   5,644


## Sample Output Inspection

In [173]:
import random

def print_record(record: dict, label: str = ""):
    print(f"\n{'='*60}")
    if label:
        print(f" {label}")
    print(f" ID : {record.get('ID')}")
    print(f" Source : {record.get('Source')}")
    print(f" Text : {str(record.get('Text',''))[:200]}...")
    print(f" Subjectivity: {record.get('Subjectivity')} (score={record.get('Subjectivity_Score')})")
    print(f" Polarity : {record.get('Polarity')} (score={record.get('Polarity_Score')})")
    print(f" Route : {record.get('Ensemble_Route')}")
    print(f" VADER : {record.get('VADER_Label')} ({record.get('VADER_Compound')})")
    print(f" SenticNet : {record.get('SenticNet_Label')} (cov={record.get('SenticNet_Coverage')})")
    print(f" Transformer : {record.get('Transformer_Label')} (conf={record.get('Transformer_Confidence')})")

# Sample 3 random posts
sample_posts = random.sample(output_posts, min(3, len(output_posts)))
for i, post in enumerate(sample_posts):
    print_record(post, f"SAMPLE POST {i+1}")
    if post.get("Comments"):
        print_record(post["Comments"][0], f"  └─ FIRST COMMENT")


 SAMPLE POST 1
 ID : 35136827
 Source : HackerNews
 Text : ...
 Subjectivity: objective (score=0.0)
 Polarity : neutral (score=0.0)
 Route : skipped_objective
 VADER : neutral (0.0)
 SenticNet : neutral (cov=0)
 Transformer : neutral (conf=0.0)

   └─ FIRST COMMENT
 ID : None
 Source : HackerNews
 Text : so thankful that this space keeps evolving.modern vanilla JS/TS isn't so bad but as a backend engineer I just cannot stay up with all the associated tooling required for building, packaging, etc. with...
 Subjectivity: subjective (score=0.3018)
 Polarity : positive (score=0.5225)
 Route : short
 VADER : positive (0.5168)
 SenticNet : positive (cov=8)
 Transformer : negative (conf=0.3828)

 SAMPLE POST 2
 ID : 1rf0rd3
 Source : Reddit
 Text : Does anyone only play raptors? I am really enjoying this mode. I understand it's essentially just 'one AI' but it's basically its own game mode.  I really enjoy defense and coop games, coming from SC2...
 Subjectivity: subjective (score=0.6112)
 P